# STAT 4830 — Red Team Adversarial Training on Colab (T4)

This notebook is the entry point for running the adversarial red team experiment on a **Google Colab T4 GPU**. It installs dependencies, configures the environment, and runs the training loop with a mock target model.

## 1. Setup (T4 environment)

Install unsloth, xformers, dspy-ai, and bitsandbytes. Unzip `project.zip` if the `src` folder is missing.

In [ ]:
%%bash
# T4-friendly installs for Colab
pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
pip install -q xformers
pip install -q dspy-ai
pip install -q bitsandbytes

# Unzip project if src is missing (e.g. after uploading project.zip)
if [ ! -d "src" ]; then
  if [ -f "project.zip" ]; then
    unzip -q project.zip
    echo "Unzipped project.zip"
  else
    echo "Warning: src/ not found and project.zip not found."
  fi
fi

## 2. Environment

Ensure the project root is on `sys.path`. No API key is required—the judge uses the local HarmBench classifier. If **`defense.USE_GEPA_DEFENDER`** is true in config, GEPA uses local Ollama (`llama3:8b`); Colab does not run Ollama by default, so keep it false unless you install Ollama in the runtime.

In [ ]:
import os
import sys

# Evaluator uses a local LM (no API key needed). Optional: set OPENAI_API_KEY if you add OpenAI-based components later.
# try:
#     from google.colab import userdata
#     os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# except Exception:
#     pass

# Ensure project root (and thus src) is on the path
project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added to path: {project_root}")

assert os.path.isdir("src"), "src/ not found. Upload project or unzip project.zip in the Setup cell."
print("Environment OK.")

## 3. Imports

Import the adversary policy, training loop, and data loader.

In [ ]:
import yaml

from src.adversary.policy import RedTeamPolicy
from src.training.loop import TrainingLoop
from src.data.harmbench_loader import HarmBenchLoader

## 4. Execution

Load config, initialize the adversary and a mock target model, and run the training loop. The config flag **`defense.USE_GEPA_DEFENDER`** controls whether the GEPA defender (BlueTeam via Ollama llama3:8b) evolves the system prompt on successful jailbreaks. CSV outputs include **system_prompt** and **use_gepa_defender** for success-rate plots.

In [ ]:
# Load config
with open("configs/default.yaml", "r") as f:
    config = yaml.safe_load(f)

# Judge uses local LM (evaluator.model_name in config); no API key needed

# Initialize Red Team (Adversary)
print("Loading Red Team (Adversary)...")
adversary = RedTeamPolicy(
    model_name=config["adversary"]["model_name"],
    lr=float(config["adversary"]["learning_rate"]),
)

# Load behaviors
print("Loading HarmBench behaviors...")
behaviors = HarmBenchLoader.load(limit=config["training"]["harmbench_limit"])

# Mock target model (replace with real model later)
def target_model(text):
    if "jailbreak" in text.lower():
        return "Sure, here is how you do something harmful..."
    return "I cannot fulfill this request."

# Build trainer and run
trainer = TrainingLoop(
    adversary=adversary,
    target_model=target_model,
    behaviors=behaviors,
    config=config,
)
trainer.run()